### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Juniors build models that predict outcomes. Seniors build models that predict the **incremental impact of an action**. In production, we don't want to know *"Will this customer churn?"* — we want to know *"Will sending this customer a retention offer **reduce** their probability of churning?"* The first is prediction. The second is causal inference.

**Mental Model:**
- **Predictive ML:** $P(Y|X)$ — What is the outcome given the features?
- **Causal ML:** $P(Y|do(T=1)) - P(Y|do(T=0))$ — What is the *change* in outcome if we *intervene*?
- **Uplift Modeling:** $\tau(x) = E[Y(1) - Y(0) | X = x]$ — What is the *individual-level* treatment effect?

**Common Junior Pitfall:** Confusing $P(Y|T=1)$ (conditional probability) with $P(Y|do(T=1))$ (interventional probability). The first is what we observe. The second is what *would happen* if we forced the treatment. They are only equal when there are no confounders — which almost never happens in the real world.

---


## 1. The Causal Revolution — DAGs, Confounders, and Do-Calculus

### 1.1 Directed Acyclic Graphs (DAGs)

A **DAG** is a graphical model that encodes causal assumptions about how variables generate data. Each node is a variable, each directed edge represents a direct causal effect.

Consider a marketing campaign example:

```
    Customer Wealth (W)
       /           \
      v             v
  Treatment (T)   Outcome (Y)
  (Email Sent)    (Purchase)
```

Here, **Wealth (W)** is a **Confounder**: it causes *both* the treatment (wealthy customers get more emails) and the outcome (wealthy customers buy more). If we naively estimate $E[Y|T=1] - E[Y|T=0]$, we get a biased estimate because the treatment and control groups differ systematically.

### 1.2 Judea Pearl's Do-Calculus

Pearl's do-operator $do(T=t)$ represents a **surgical intervention** — we *set* the treatment to $t$, breaking all incoming arrows to $T$ in the DAG.

**The Adjustment Formula (Backdoor Criterion):**

If a set of variables $Z$ blocks all backdoor paths from $T$ to $Y$, then:

$$P(Y|do(T=t)) = \sum_z P(Y|T=t, Z=z) \cdot P(Z=z)$$

This is the mathematical foundation that allows us to estimate causal effects from observational data — *if* we can identify and measure the right confounders.

### 1.3 The Three Elemental Junctions

| Junction Type | Structure | Conditioning Effect |
|:---|:---|:---|
| **Chain** (Mediation) | $A \to B \to C$ | Conditioning on $B$ *blocks* the path |
| **Fork** (Confounding) | $A \leftarrow B \to C$ | Conditioning on $B$ *blocks* the path |
| **Collider** | $A \to B \leftarrow C$ | Conditioning on $B$ *opens* the path (Berkson's Paradox) |

> **⚠️ Critical Insight:** Conditioning on a **collider** *introduces* spurious correlation. This is why "controlling for everything" is dangerous — it can create bias where none existed.


## 📑 Table of Contents

  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [1. The Causal Revolution — DAGs, Confounders, and Do-Calculus](#1-the-causal-revolution-dags-confounders-and-do-calculus)
  * [1.1 Directed Acyclic Graphs (DAGs)](#11-directed-acyclic-graphs-dags)
  * [1.2 Judea Pearl's Do-Calculus](#12-judea-pearls-do-calculus)
  * [1.3 The Three Elemental Junctions](#13-the-three-elemental-junctions)
* [2. Propensity Score Matching (PSM)](#2-propensity-score-matching-psm)
  * [The Core Idea](#the-core-idea)
  * [Mathematical Guarantee](#mathematical-guarantee)
* [3. Inverse Probability of Treatment Weighting (IPTW)](#3-inverse-probability-of-treatment-weighting-iptw)
  * [Why IPTW over PSM?](#why-iptw-over-psm)
  * [The Horvitz-Thompson Estimator](#the-horvitz-thompson-estimator)
* [4. Uplift Modeling — Meta-Learners for CATE Estimation](#4-uplift-modeling-meta-learners-for-cate-estimation)
  * [Beyond ATE: Why Heterogeneous Effects Matter](#beyond-ate-why-heterogeneous-effects-matter)
  * [The Four Quadrants of Treatment Response](#the-four-quadrants-of-treatment-response)
  * [4.1 The S-Learner (Single Model)](#41-the-s-learner-single-model)
  * [4.2 The T-Learner (Two Models)](#42-the-t-learner-two-models)
  * [4.3 The X-Learner (Cross-Learner)](#43-the-x-learner-cross-learner)
* [5. Production Pipeline — EconML and Doubly Robust Estimation](#5-production-pipeline-econml-and-doubly-robust-estimation)
  * [The Doubly Robust Estimator](#the-doubly-robust-estimator)
  * [Microsoft's EconML](#microsofts-econml)
* [✅ Knowledge Check](#knowledge-check)
  * [Q1: Observational vs Interventional](#q1-observational-vs-interventional)
  * [Q2: Why is the X-Learner preferred for imbalanced treatments?](#q2-why-is-the-x-learner-preferred-for-imbalanced-treatments)
  * [Q3: Why is the Doubly Robust estimator the production standard?](#q3-why-is-the-doubly-robust-estimator-the-production-standard)
* [🔨 Practical Practice](#practical-practice)
  * [Tier 1: Beginner](#tier-1-beginner)
  * [Tier 2: Intermediate](#tier-2-intermediate)
  * [Tier 3: Advanced](#tier-3-advanced)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
np.random.seed(42)

# ============================================================
# Simulate Observational Data with a Known Confounder
# ============================================================
# Ground truth: TRUE Average Treatment Effect (ATE) = 2.0
# But the confounder 'wealth' biases naive estimates upward.

n = 5000
wealth = np.random.normal(50, 15, n)                     # Confounder
prob_treatment = 1 / (1 + np.exp(-(wealth - 50) / 10))   # Wealthy -> more likely treated
treatment = np.random.binomial(1, prob_treatment)         # Binary treatment

TRUE_ATE = 2.0
outcome = 10 + 0.5 * wealth + TRUE_ATE * treatment + np.random.normal(0, 5, n)

df = pd.DataFrame({'wealth': wealth, 'treatment': treatment, 'outcome': outcome})

# Naive estimate (biased by confounding)
naive_ate = df[df.treatment == 1]['outcome'].mean() - df[df.treatment == 0]['outcome'].mean()
print(f"TRUE ATE:  {TRUE_ATE:.2f}")
print(f"Naive ATE: {naive_ate:.2f}  <-- Biased upward by wealth confounding")
print(f"Bias:      {naive_ate - TRUE_ATE:+.2f}")

"""
What just happened?
We simulated a dataset where the TRUE causal effect of treatment is exactly 2.0.
But the naive comparison (treated mean - control mean) yields ~5-7, because wealthy
customers are both more likely to be treated AND more likely to have high outcomes.
The confounder inflates our estimate. The rest of this notebook fixes this.
"""

## 2. Propensity Score Matching (PSM)

### The Core Idea

The **Propensity Score** is the probability of receiving treatment given observed covariates:

$$e(x) = P(T = 1 | X = x)$$

**Rosenbaum & Rubin (1983)** proved that if treatment assignment is **Strongly Ignorable** given $X$ (i.e., no unmeasured confounders), then it is also ignorable given the scalar $e(X)$. This reduces a high-dimensional matching problem to a 1D problem.

**PSM Algorithm:**
1. Estimate $e(x)$ using logistic regression (or any classifier).
2. For each treated unit, find the closest control unit by propensity score.
3. Estimate ATE from the matched pairs.

### Mathematical Guarantee

Under the **Conditional Independence Assumption** (CIA): $(Y(0), Y(1)) \perp T | X$

The ATE is identified as:
$$\tau_{ATE} = E\left[\frac{TY}{e(X)} - \frac{(1-T)Y}{1-e(X)}\right]$$


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors

# ============================================================
# Step 1: Estimate Propensity Scores
# ============================================================
X_confounders = df[['wealth']].values
ps_model = LogisticRegression()
ps_model.fit(X_confounders, df['treatment'])
df['propensity_score'] = ps_model.predict_proba(X_confounders)[:, 1]

# Visualize propensity score distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df[df.treatment == 1]['propensity_score'], bins=30, alpha=0.7, 
             color='#e74c3c', label='Treated', density=True)
axes[0].hist(df[df.treatment == 0]['propensity_score'], bins=30, alpha=0.7, 
             color='#3498db', label='Control', density=True)
axes[0].set_xlabel('Propensity Score')
axes[0].set_title('Before Matching: Propensity Score Distributions')
axes[0].legend()

# ============================================================
# Step 2: 1:1 Nearest-Neighbor Matching on Propensity Score
# ============================================================
treated_idx = df[df.treatment == 1].index
control_idx = df[df.treatment == 0].index

nn = NearestNeighbors(n_neighbors=1, metric='euclidean')
nn.fit(df.loc[control_idx, ['propensity_score']].values)
distances, indices = nn.kneighbors(df.loc[treated_idx, ['propensity_score']].values)

matched_control_idx = control_idx[indices.flatten()]
matched_treated = df.loc[treated_idx]
matched_control = df.loc[matched_control_idx]

# Visualize after matching
axes[1].hist(matched_treated['propensity_score'], bins=30, alpha=0.7, 
             color='#e74c3c', label='Treated', density=True)
axes[1].hist(matched_control['propensity_score'], bins=30, alpha=0.7, 
             color='#3498db', label='Matched Control', density=True)
axes[1].set_xlabel('Propensity Score')
axes[1].set_title('After Matching: Balanced Distributions')
axes[1].legend()
plt.tight_layout()
plt.show()

# ============================================================
# Step 3: Estimate ATE from Matched Pairs
# ============================================================
psm_ate = matched_treated['outcome'].values.mean() - matched_control['outcome'].values.mean()
print(f"\nTRUE ATE:  {TRUE_ATE:.2f}")
print(f"Naive ATE: {naive_ate:.2f}")
print(f"PSM ATE:   {psm_ate:.2f}  <-- Much closer to truth!")

# Covariate balance check
print(f"\nCovariate Balance (Wealth):")
print(f"  Before: Treated={df[df.treatment==1]['wealth'].mean():.1f}, "
      f"Control={df[df.treatment==0]['wealth'].mean():.1f}")
print(f"  After:  Treated={matched_treated['wealth'].mean():.1f}, "
      f"Control={matched_control['wealth'].mean():.1f}")

"""
What just happened?
We estimated each unit's propensity score, then matched each treated unit to its 
nearest control neighbor. After matching, the wealth distributions are balanced, 
removing the confounding bias. The PSM ATE estimate is now close to the true 2.0.
"""

## 3. Inverse Probability of Treatment Weighting (IPTW)

### Why IPTW over PSM?

PSM discards unmatched units, reducing statistical power. **IPTW** uses *all* data by re-weighting each observation to create a **pseudo-population** where treatment is independent of confounders.

### The Horvitz-Thompson Estimator

Each unit receives a weight inversely proportional to its probability of receiving the treatment it actually received:

$$w_i = \frac{T_i}{e(X_i)} + \frac{1 - T_i}{1 - e(X_i)}$$

The **IPTW ATE** is then:

$$\hat{\tau}_{IPTW} = \frac{1}{n}\sum_{i=1}^{n} \frac{T_i Y_i}{e(X_i)} - \frac{1}{n}\sum_{i=1}^{n} \frac{(1-T_i) Y_i}{1 - e(X_i)}$$

**Intuition:** A treated unit with a *low* propensity score is "surprising" — it represents many similar untreated units, so it gets a high weight. This balances the pseudo-population.

> **⚠️ Practical Warning:** Extreme propensity scores (near 0 or 1) create extreme weights, inflating variance. **Stabilized weights** or **weight trimming** at the 1st/99th percentile are essential in production.


In [ ]:
# ============================================================
# IPTW: Inverse Probability of Treatment Weighting
# ============================================================
ps = df['propensity_score'].values
t = df['treatment'].values
y = df['outcome'].values

# Clip propensity scores to avoid extreme weights
ps_clipped = np.clip(ps, 0.05, 0.95)

# Horvitz-Thompson estimator
iptw_treated = np.mean((t * y) / ps_clipped)
iptw_control = np.mean(((1 - t) * y) / (1 - ps_clipped))
iptw_ate = iptw_treated - iptw_control

# Stabilized weights (Hajek estimator — normalized)
w1 = t / ps_clipped
w0 = (1 - t) / (1 - ps_clipped)
hajek_ate = (np.sum(w1 * y) / np.sum(w1)) - (np.sum(w0 * y) / np.sum(w0))

print(f"TRUE ATE:     {TRUE_ATE:.2f}")
print(f"Naive ATE:    {naive_ate:.2f}")
print(f"PSM ATE:      {psm_ate:.2f}")
print(f"IPTW ATE (HT):  {iptw_ate:.2f}")
print(f"IPTW ATE (Hajek): {hajek_ate:.2f}  <-- Stabilized, preferred in practice")

# Visualize weight distribution
weights = np.where(t == 1, 1 / ps_clipped, 1 / (1 - ps_clipped))
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(weights, bins=50, color='#9b59b6', alpha=0.7, edgecolor='white')
ax.set_xlabel('IPTW Weight')
ax.set_ylabel('Count')
ax.set_title('Distribution of IPTW Weights (Clipped at [0.05, 0.95])')
ax.axvline(np.median(weights), color='red', linestyle='--', label=f'Median: {np.median(weights):.1f}')
ax.legend()
plt.tight_layout()
plt.show()

"""
What just happened?
Instead of discarding data (PSM), IPTW re-weights every observation. The Hajek 
(stabilized) estimator is preferred because it normalizes weights, reducing 
variance from extreme propensity scores. Both IPTW estimates recover the true ATE.
"""

## 4. Uplift Modeling — Meta-Learners for CATE Estimation

### Beyond ATE: Why Heterogeneous Effects Matter

The ATE tells us the *average* effect across the population. But treatment effects are rarely uniform. The **Conditional Average Treatment Effect (CATE)** captures *individual-level* variation:

$$\tau(x) = E[Y(1) - Y(0) | X = x]$$

This enables **precision targeting**: treat only the individuals for whom the intervention has a *positive* incremental effect.

### The Four Quadrants of Treatment Response

| Quadrant | $Y(0)$ | $Y(1)$ | $\tau(x)$ | Action |
|:---|:---|:---|:---|:---|
| **Persuadables** | Low | High | Positive | ✅ Target these |
| **Sure Things** | High | High | Zero | ❌ Don't waste resources |
| **Lost Causes** | Low | Low | Zero | ❌ Don't waste resources |
| **Sleeping Dogs** | High | Low | Negative | ⚠️ Treatment *hurts* — avoid! |

> **🎯 Production Goal:** Uplift models identify the **Persuadables** — the only group where treatment creates incremental value.

---

### 4.1 The S-Learner (Single Model)

**Approach:** Train a single model $\hat{\mu}(X, T)$ on features AND treatment indicator. Estimate CATE as:

$$\hat{\tau}(x) = \hat{\mu}(x, T=1) - \hat{\mu}(x, T=0)$$

**Strength:** Simple, uses all data.
**Weakness:** The model may "regularize away" the treatment effect if $T$ is just one of many features.

### 4.2 The T-Learner (Two Models)

**Approach:** Train two separate models — $\hat{\mu}_1(X)$ on treated units and $\hat{\mu}_0(X)$ on control units:

$$\hat{\tau}(x) = \hat{\mu}_1(x) - \hat{\mu}_0(x)$$

**Strength:** Each model is optimized for its subgroup.
**Weakness:** If treatment/control group sizes are very unbalanced, one model may be much noisier.

### 4.3 The X-Learner (Cross-Learner)

**Approach (Kúnzel et al., 2019):** A three-stage procedure that handles treatment group imbalance:

1. **Stage 1:** Train $\hat{\mu}_0$ and $\hat{\mu}_1$ (like T-Learner).
2. **Stage 2:** Impute individual treatment effects:
   - For treated: $\tilde{D}_1^i = Y_i^1 - \hat{\mu}_0(X_i^1)$
   - For control: $\tilde{D}_0^i = \hat{\mu}_1(X_i^0) - Y_i^0$
3. **Stage 3:** Train CATE models on imputed effects, then combine:
   $$\hat{\tau}(x) = g(x) \cdot \hat{\tau}_0(x) + (1 - g(x)) \cdot \hat{\tau}_1(x)$$
   where $g(x) = e(x)$ (the propensity score) serves as the weighting function.

**Strength:** Excels when one group is much larger — leverages the larger group to improve estimates for the smaller group.


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

# ============================================================
# Generate Heterogeneous Treatment Effect Data
# ============================================================
# The TRUE CATE varies by individual: tau(x) = 0.5 * age - 10
# Young people (age < 20) have NEGATIVE treatment effects (sleeping dogs)
# Older people (age > 20) have POSITIVE treatment effects (persuadables)

np.random.seed(42)
n = 3000
age = np.random.uniform(10, 60, n)
income = np.random.normal(50, 15, n)
X_uplift = np.column_stack([age, income])

# Treatment assignment (biased by income)
ps_true = 1 / (1 + np.exp(-(income - 50) / 10))
T = np.random.binomial(1, ps_true)

# True individual treatment effect (heterogeneous!)
true_cate = 0.5 * age - 10  # Crosses zero at age=20

# Observed outcomes
Y0 = 20 + 0.3 * income + np.random.normal(0, 3, n)       # Potential outcome without treatment
Y1 = Y0 + true_cate                                       # Potential outcome with treatment
Y_obs = T * Y1 + (1 - T) * Y0                             # Observed (factual) outcome

print(f"True ATE: {true_cate.mean():.2f}")
print(f"Fraction with negative CATE (sleeping dogs): {(true_cate < 0).mean():.1%}")
print(f"Fraction with positive CATE (persuadables):  {(true_cate > 0).mean():.1%}")

In [ ]:
# ============================================================
# S-Learner Implementation
# ============================================================
X_with_T = np.column_stack([X_uplift, T])
s_model = GradientBoostingRegressor(n_estimators=200, max_depth=4, random_state=42)
s_model.fit(X_with_T, Y_obs)

# Predict CATE: difference in predictions when T=1 vs T=0
X_t1 = np.column_stack([X_uplift, np.ones(n)])
X_t0 = np.column_stack([X_uplift, np.zeros(n)])
cate_s = s_model.predict(X_t1) - s_model.predict(X_t0)

# ============================================================
# T-Learner Implementation
# ============================================================
model_1 = GradientBoostingRegressor(n_estimators=200, max_depth=4, random_state=42)
model_0 = GradientBoostingRegressor(n_estimators=200, max_depth=4, random_state=42)
model_1.fit(X_uplift[T == 1], Y_obs[T == 1])
model_0.fit(X_uplift[T == 0], Y_obs[T == 0])
cate_t = model_1.predict(X_uplift) - model_0.predict(X_uplift)

# ============================================================
# X-Learner Implementation
# ============================================================
# Stage 1: Reuse T-Learner models (model_0, model_1)

# Stage 2: Impute individual treatment effects
D1 = Y_obs[T == 1] - model_0.predict(X_uplift[T == 1])  # Treated: actual - counterfactual
D0 = model_1.predict(X_uplift[T == 0]) - Y_obs[T == 0]  # Control: counterfactual - actual

# Stage 3: Train CATE models on imputed effects
tau_model_1 = GradientBoostingRegressor(n_estimators=200, max_depth=4, random_state=42)
tau_model_0 = GradientBoostingRegressor(n_estimators=200, max_depth=4, random_state=42)
tau_model_1.fit(X_uplift[T == 1], D1)
tau_model_0.fit(X_uplift[T == 0], D0)

# Combine with propensity score weighting
ps_est = LogisticRegression().fit(X_uplift, T).predict_proba(X_uplift)[:, 1]
cate_x = ps_est * tau_model_0.predict(X_uplift) + (1 - ps_est) * tau_model_1.predict(X_uplift)

print("Meta-Learner ATE Estimates vs True ATE:")
print(f"  True ATE:     {true_cate.mean():.2f}")
print(f"  S-Learner:    {cate_s.mean():.2f}")
print(f"  T-Learner:    {cate_t.mean():.2f}")
print(f"  X-Learner:    {cate_x.mean():.2f}")

In [ ]:
# ============================================================
# Visualization: CATE Estimates vs Ground Truth
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sort_idx = np.argsort(age)

for ax, cate_est, name, color in zip(
    axes,
    [cate_s, cate_t, cate_x],
    ['S-Learner', 'T-Learner', 'X-Learner'],
    ['#e74c3c', '#3498db', '#2ecc71']
):
    ax.scatter(age, cate_est, alpha=0.15, s=8, color=color, label=f'{name} Estimate')
    ax.plot(age[sort_idx], true_cate[sort_idx], color='black', linewidth=2, label='True CATE')
    ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel('Age')
    ax.set_ylabel('CATE (Treatment Effect)')
    ax.set_title(name)
    ax.legend(loc='upper left')

fig.suptitle('Meta-Learner CATE Estimates vs Ground Truth', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# RMSE comparison
for cate_est, name in zip([cate_s, cate_t, cate_x], ['S-Learner', 'T-Learner', 'X-Learner']):
    rmse = np.sqrt(np.mean((cate_est - true_cate) ** 2))
    print(f"{name} CATE RMSE: {rmse:.3f}")

"""
What just happened?
We implemented all three meta-learners from scratch. The scatter plots show each 
learner's CATE estimate (colored) vs the ground truth (black line). The X-Learner 
typically produces the tightest fit, especially when treatment groups are imbalanced,
because it cross-pollinates information between groups via imputed treatment effects.
"""

## 5. Production Pipeline — EconML and Doubly Robust Estimation

### The Doubly Robust Estimator

The **Doubly Robust (DR)** estimator combines outcome modeling AND propensity weighting. It is consistent if *either* the outcome model *or* the propensity model is correctly specified (but not necessarily both):

$$\hat{\tau}_{DR} = \frac{1}{n} \sum_{i=1}^n \left[ \hat{\mu}_1(X_i) - \hat{\mu}_0(X_i) + \frac{T_i(Y_i - \hat{\mu}_1(X_i))}{e(X_i)} - \frac{(1-T_i)(Y_i - \hat{\mu}_0(X_i))}{1 - e(X_i)} \right]$$

This is the gold standard in production because it provides **double insurance** against model misspecification.

### Microsoft's EconML

**EconML** provides production-grade implementations of heterogeneous treatment effect estimators with confidence intervals, built on top of scikit-learn.


In [ ]:
try:
    from econml.dml import LinearDML, CausalForestDML
    from econml.metalearners import TLearner, SLearner, XLearner
except ImportError:
    print("Installing EconML...")
    !pip install -q econml
    from econml.dml import LinearDML, CausalForestDML
    from econml.metalearners import TLearner, SLearner, XLearner

from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier

# ============================================================
# EconML: Double Machine Learning (DML) with Causal Forest
# ============================================================
# DML uses cross-fitting to avoid overfitting bias in the nuisance models

cf_dml = CausalForestDML(
    model_y=GradientBoostingRegressor(n_estimators=100, max_depth=3),
    model_t=GradientBoostingClassifier(n_estimators=100, max_depth=3),
    n_estimators=200,
    random_state=42
)

# W = confounders, X = effect modifiers (heterogeneity drivers)
W = income.reshape(-1, 1)     # Confounder
X_het = age.reshape(-1, 1)    # Heterogeneity variable

cf_dml.fit(Y_obs, T, X=X_het, W=W)

# Get CATE estimates with confidence intervals
cate_dml = cf_dml.effect(X_het)
cate_lb, cate_ub = cf_dml.effect_interval(X_het, alpha=0.05)

print(f"CausalForestDML ATE: {cate_dml.mean():.2f}")
print(f"True ATE:            {true_cate.mean():.2f}")

In [ ]:
# ============================================================
# Final Visualization: CausalForestDML with Confidence Intervals
# ============================================================
sort_idx = np.argsort(age)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(age[sort_idx], true_cate[sort_idx], color='black', linewidth=2.5, 
        label='True CATE: $\\tau(x) = 0.5 \\cdot age - 10$', zorder=5)
ax.scatter(age, cate_dml, alpha=0.15, s=10, color='#2ecc71', label='CausalForestDML Estimate')
ax.fill_between(age[sort_idx], cate_lb[sort_idx], cate_ub[sort_idx], 
                alpha=0.15, color='#2ecc71', label='95% Confidence Interval')
ax.axhline(0, color='red', linestyle='--', alpha=0.7, label='Zero Effect Line')
ax.axvline(20, color='orange', linestyle=':', alpha=0.7, label='Crossover Point (age=20)')

# Annotate regions
ax.annotate('SLEEPING DOGS\n(Treatment Hurts)', xy=(14, -4), fontsize=11, 
            color='#e74c3c', fontweight='bold', ha='center')
ax.annotate('PERSUADABLES\n(Treatment Helps)', xy=(45, 10), fontsize=11, 
            color='#27ae60', fontweight='bold', ha='center')

ax.set_xlabel('Age', fontsize=12)
ax.set_ylabel('CATE (Individual Treatment Effect)', fontsize=12)
ax.set_title('CausalForestDML: Heterogeneous Treatment Effect Recovery\nwith 95% Confidence Intervals', 
             fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=10)
plt.tight_layout()
plt.show()

# CATE RMSE
rmse_dml = np.sqrt(np.mean((cate_dml - true_cate) ** 2))
print(f"\nCausalForestDML CATE RMSE: {rmse_dml:.3f}")

# Policy evaluation: target only persuadables
treated_persuadables = (cate_dml > 0).sum()
total_positive_cate = (true_cate > 0).sum()
print(f"\nPolicy Evaluation:")
print(f"  Model recommends treating {treated_persuadables} / {n} individuals")
print(f"  True persuadables: {total_positive_cate} / {n}")
print(f"  Expected uplift from targeted policy: {cate_dml[cate_dml > 0].sum():.1f}")
print(f"  Expected uplift from treat-all policy: {cate_dml.sum():.1f}")

"""
What just happened?
EconML's CausalForestDML recovered the heterogeneous treatment effect curve with 
confidence intervals. The model correctly identifies the crossover at age~20: 
treating younger individuals hurts outcomes (sleeping dogs), while targeting older 
individuals creates genuine uplift (persuadables). The targeted policy dramatically 
outperforms treat-all by avoiding negative treatment effects.
"""

---
## ✅ Knowledge Check

### Q1: Observational vs Interventional
<details><summary>👉 Answer</summary>
$P(Y|T=1)$ conditions on observed treatment — it includes confounding bias. $P(Y|do(T=1))$ simulates an intervention by breaking all incoming edges to $T$ in the DAG, isolating the causal effect. They are only equal when there are no confounders (backdoor paths are absent).
</details>

### Q2: Why is the X-Learner preferred for imbalanced treatments?
<details><summary>👉 Answer</summary>
When one treatment group is much smaller, the T-Learner's model for that group has high variance due to limited data. The X-Learner mitigates this by using the larger group's model to impute counterfactual outcomes for the smaller group, then training a CATE model on these imputed effects. The propensity-weighted combination further balances the estimates.
</details>

### Q3: Why is the Doubly Robust estimator the production standard?
<details><summary>👉 Answer</summary>
It is consistent if EITHER the outcome model OR the propensity model is correctly specified. In practice, we never know which model is right, so double robustness provides insurance against partial misspecification — a critical property for high-stakes business decisions.
</details>


---
## 🔨 Practical Practice

### Tier 1: Beginner
1. Draw the DAG for the following scenario: "Ice cream sales and drowning deaths are positively correlated." Identify the confounder and explain why banning ice cream would NOT reduce drownings.
2. Using the simulated dataset above, compute the naive ATE, PSM ATE, and IPTW ATE. Verify that PSM and IPTW recover the true effect while the naive estimate does not.

### Tier 2: Intermediate
1. **Collider Bias:** Simulate a collider structure ($A \to C \leftarrow B$) where $A$ and $B$ are independent. Show that conditioning on $C$ induces a spurious correlation between $A$ and $B$ (Berkson's Paradox).
2. **Meta-Learner Comparison:** Modify the true CATE function to $\tau(x) = \sin(age / 5)$ (non-linear, oscillating). Re-run all three meta-learners and compare RMSE. Which learner handles non-linearity best?

### Tier 3: Advanced
1. **Sensitivity Analysis:** Implement the E-value framework (VanderWeele & Ding, 2017) to quantify how strong an unmeasured confounder would need to be to explain away the observed treatment effect. Apply it to the PSM result.
2. **Double ML from Scratch:** Implement the Chernozhukov et al. (2018) Double/Debiased ML algorithm using cross-fitting. Compare your implementation's ATE estimate and standard error against EconML's `LinearDML`.
